# Effective Access: Two-Tier RBAC — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/system-design/11-1/effective-access-two-tier-rbac.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/system-design/11-1/effective-access-two-tier-rbac.ipynb)

Companion to: [Effective Access: Combining Two Levels of Role-Based Permissions](https://anacodicai.org/learn/system-design/11-1/effective-access-two-tier-rbac)


## What you'll build

1. An `effective_access` function that checks an **org-wide role** first, and
   only falls through to a **resource-specific role** if the org role doesn't
   already settle the question.
2. Seven scenarios that exercise every combination: org owner with no
   resource role, plain member with a resource role, plain member with none,
   and a non-member entirely.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
print("ready — standard library only")

ready — standard library only


## Step 1 — two role systems, one resource

`ORG_ROLE_GRANTS_ALL` are the org-wide roles that should be able to act on
*every* resource in the org, without needing a matching resource-level grant.
`RESOURCE_ROLE_PERMISSIONS` are what a resource-specific role can do, and
nothing more.

In [2]:
ORG_ROLE_GRANTS_ALL = {"owner", "admin"}

RESOURCE_ROLE_PERMISSIONS = {
    "instructor": {"view", "edit_content", "grade", "manage_resource_members"},
    "ta":         {"view", "grade"},
    "student":    {"view", "submit"},
}
print("role tables ready")

role tables ready


## Step 2 — the decision, in the right order

Org-wide authority is checked FIRST. Only if it doesn't apply does the
function fall through to the resource-specific role — and if there's no org
membership at all, it denies immediately without even looking at the
resource role.

In [3]:
def effective_access(org_role, resource_role, action):
    if org_role is None:
        return False  # not in the organization at all -> no access, period
    if org_role in ORG_ROLE_GRANTS_ALL:
        return True   # org owner/admin can act on every resource in the org
    if resource_role is None:
        return False  # plain org member, no resource-specific role -> nothing
    return action in RESOURCE_ROLE_PERMISSIONS.get(resource_role, set())

print("effective_access() ready")

effective_access() ready


## Step 3 — run all seven scenarios

**Try it:** add an 8th scenario — an org `admin` who ALSO happens to hold a
`"student"` resource role — and confirm the org-wide grant still wins (the
function returns `True` for `manage_resource_members` regardless of the
`"student"` role, because the org check runs first).

In [4]:
CHECKS = [
    ("org owner, no resource role",   "owner",  None,        "manage_resource_members"),
    ("org admin, no resource role",   "admin",  None,        "edit_content"),
    ("plain org member, instructor role on THIS resource", "member", "instructor", "grade"),
    ("plain org member, student role on THIS resource",    "member", "student",    "manage_resource_members"),
    ("plain org member, student role on THIS resource",    "member", "student",    "submit"),
    ("plain org member, NO role on this resource",         "member", None,         "view"),
    ("not an org member at all",                            None,    "instructor", "view"),
]

print(f"{'scenario':56}{'action':26}{'access'}")
for label, org_role, resource_role, action in CHECKS:
    result = effective_access(org_role, resource_role, action)
    print(f"{label:56}{action:26}{result}")

scenario                                                action                    access
org owner, no resource role                             manage_resource_members   True
org admin, no resource role                             edit_content              True
plain org member, instructor role on THIS resource      grade                     True
plain org member, student role on THIS resource         manage_resource_members   False
plain org member, student role on THIS resource         submit                    True
plain org member, NO role on this resource              view                      False
not an org member at all                                view                      False


## Step 4 — the two directions this could go wrong

An org owner should NOT need a resource role. A non-member should NEVER get
access even if a stale resource-role record exists for them.

In [5]:
assert effective_access("admin", None, "manage_resource_members") is True
assert effective_access("member", None, "view") is False
assert effective_access("member", "student", "manage_resource_members") is False
assert effective_access("member", "student", "submit") is True
assert effective_access(None, "instructor", "view") is False
print("all 5 asserts passed: org-wide grants work without a resource role, "
      "and no org membership means no access regardless of resource role")

all 5 asserts passed: org-wide grants work without a resource role, and no org membership means no access regardless of resource role


## Try it yourself

- Add a rule that should override BOTH tiers — a platform-wide suspension —
  and notice it doesn't fit naturally inside `effective_access` at all; it
  needs its own check that runs *before* this function, returning `False`
  immediately if suspended.
- Add a third tier (say, a "team" inside a resource) and see how many new
  precedence decisions you have to make explicit and test.